# AgentFence.AI — Module 1: Synthetic Scenario Generator

## Objective

Create fictional consumer scenarios that AgentFence.AI can use to study whether an access request fits the user’s intended task.

Each scenario includes:

- What the user wants to do.
- What access or action the website, application, or AI assistant requests.
- The resource and scope involved.
- Whether the request is temporary or ongoing.
- A human-written expected decision: allow, warn, or block.

## Data Sources

This notebook uses the two files created by Module 0:

- `project_definition.json`
- `action_taxonomy.csv`

Attach Module 0’s notebook output files as inputs before running this notebook.

## Scenario Categories

The scenarios cover:

- File uploads and file access.
- Public and private sharing.
- Account permissions.
- Browser extension permissions.
- Artificial intelligence (AI) assistant and agent actions.

All examples use fictional data. They do not connect to real accounts or use real personal files.

## Expected Decisions

The expected decision is a human-written reference label for later evaluation:

- **Allow:** The requested access appears limited to the task.
- **Warn:** The request may be broader than needed, and the user should review it.
- **Block:** The request includes a serious mismatch or an action the user did not approve.

These labels are evaluation references. Later modules will compare AgentFence.AI’s decisions against them.

## Output Files

This notebook creates:

- `synthetic_scenarios.csv` — scenario data for Modules 2 and 3.
- `scenario_summary.json` — scenario counts and source-file information.

Module 2 will process the user’s task descriptions. Module 3 will process the requested permissions and actions.

In [1]:
from pathlib import Path
import csv
import json

input_root = Path("/kaggle/input")

def find_input_file(filename):
    matches = sorted(input_root.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach Module 0's notebook output files "
            "as inputs to this notebook, then run again."
        )
    return matches[0]

definition_path = find_input_file("project_definition.json")
taxonomy_path = find_input_file("action_taxonomy.csv")

with definition_path.open(encoding="utf-8") as file:
    project_definition = json.load(file)

with taxonomy_path.open(newline="", encoding="utf-8") as file:
    action_taxonomy = list(csv.DictReader(file))

valid_action_types = {row["action_type"] for row in action_taxonomy}
valid_decisions = set(project_definition["decision_labels"])

scenarios = [
    {
        "scenario_id": "S001",
        "category": "file_upload",
        "scenario_title": "Upload one résumé",
        "user_task_text": "Upload one résumé to a job application website.",
        "app_request_text": "Select one résumé file and upload it for this application.",
        "action_types": "file_upload",
        "resource": "résumé file",
        "resource_scope": "one selected file",
        "access_mode": "upload",
        "duration": "one time",
        "recipient_or_visibility": "job application website",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "false",
        "expected_decision": "allow",
        "expected_reason": "The request is limited to the one file needed for the task.",
    },
    {
        "scenario_id": "S002",
        "category": "file_upload",
        "scenario_title": "Résumé upload requests the full drive",
        "user_task_text": "Upload one résumé to a job application website.",
        "app_request_text": "Grant ongoing read access to every file in the cloud drive.",
        "action_types": "file_read;account_permission",
        "resource": "cloud drive",
        "resource_scope": "all files",
        "access_mode": "read",
        "duration": "ongoing",
        "recipient_or_visibility": "job application website",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "true",
        "expected_decision": "warn",
        "expected_reason": "Reading the entire drive is broader than uploading one résumé.",
    },
    {
        "scenario_id": "S003",
        "category": "file_upload",
        "scenario_title": "Résumé upload requests drive and contact control",
        "user_task_text": "Upload one résumé to a job application website.",
        "app_request_text": "Read and modify all drive files, access contacts, and retain account access.",
        "action_types": "file_read;file_modify;contact_access;account_permission",
        "resource": "cloud drive and contacts",
        "resource_scope": "all files and all contacts",
        "access_mode": "read and modify",
        "duration": "ongoing",
        "recipient_or_visibility": "job application website",
        "can_read": "true",
        "can_modify": "true",
        "can_share": "false",
        "persistent_access": "true",
        "expected_decision": "block",
        "expected_reason": "The request includes broad file changes and contact access unrelated to the upload.",
    },
    {
        "scenario_id": "S004",
        "category": "file_sharing",
        "scenario_title": "Share a document privately",
        "user_task_text": "Share one project document with a named colleague for review.",
        "app_request_text": "Create a private, view-only link for that colleague that expires in seven days.",
        "action_types": "file_share",
        "resource": "one project document",
        "resource_scope": "one selected document",
        "access_mode": "view only",
        "duration": "seven days",
        "recipient_or_visibility": "one named colleague",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "true",
        "persistent_access": "false",
        "expected_decision": "allow",
        "expected_reason": "The sharing scope and duration match the stated task.",
    },
    {
        "scenario_id": "S005",
        "category": "file_sharing",
        "scenario_title": "Share through an unexpired public link",
        "user_task_text": "Share one project document with a named colleague for review.",
        "app_request_text": "Create a view-only link that anyone can open with no expiration.",
        "action_types": "file_share",
        "resource": "one project document",
        "resource_scope": "one selected document",
        "access_mode": "view only",
        "duration": "no expiration",
        "recipient_or_visibility": "anyone with the link",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "true",
        "persistent_access": "true",
        "expected_decision": "warn",
        "expected_reason": "Anyone with the link may access the document, and access does not expire.",
    },
    {
        "scenario_id": "S006",
        "category": "file_sharing",
        "scenario_title": "Public editing access to a private document",
        "user_task_text": "Share a private financial document with one accountant.",
        "app_request_text": "Create a public link that allows anyone to edit the document.",
        "action_types": "file_share;file_modify",
        "resource": "financial document",
        "resource_scope": "one sensitive document",
        "access_mode": "edit",
        "duration": "no expiration",
        "recipient_or_visibility": "public link",
        "can_read": "true",
        "can_modify": "true",
        "can_share": "true",
        "persistent_access": "true",
        "expected_decision": "block",
        "expected_reason": "Public editing access exposes a sensitive document beyond the intended recipient.",
    },
    {
        "scenario_id": "S007",
        "category": "account_permissions",
        "scenario_title": "Schedule one calendar event",
        "user_task_text": "Schedule one meeting on my calendar.",
        "app_request_text": "Temporarily read and update the selected calendar to create the event.",
        "action_types": "account_permission",
        "resource": "selected calendar",
        "resource_scope": "one calendar",
        "access_mode": "read and create event",
        "duration": "one day",
        "recipient_or_visibility": "meeting attendees",
        "can_read": "true",
        "can_modify": "true",
        "can_share": "false",
        "persistent_access": "false",
        "expected_decision": "allow",
        "expected_reason": "Temporary access to the selected calendar supports the requested task.",
    },
    {
        "scenario_id": "S008",
        "category": "account_permissions",
        "scenario_title": "Calendar task requests email and contacts",
        "user_task_text": "Schedule one meeting on my calendar.",
        "app_request_text": "Read all email and contacts and keep ongoing access to the account.",
        "action_types": "file_read;contact_access;account_permission",
        "resource": "email, contacts, and calendar",
        "resource_scope": "all email and all contacts",
        "access_mode": "read",
        "duration": "ongoing",
        "recipient_or_visibility": "scheduling service",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "true",
        "expected_decision": "warn",
        "expected_reason": "Reading all email and contacts appears broader than scheduling one meeting.",
    },
    {
        "scenario_id": "S009",
        "category": "account_permissions",
        "scenario_title": "Send invitations to all contacts",
        "user_task_text": "Schedule a meeting with two selected colleagues.",
        "app_request_text": "Send invitations to every contact without showing a confirmation.",
        "action_types": "contact_access;ai_agent_action",
        "resource": "contacts and calendar",
        "resource_scope": "all contacts",
        "access_mode": "read contacts and send invitations",
        "duration": "immediate",
        "recipient_or_visibility": "all contacts",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "true",
        "persistent_access": "false",
        "expected_decision": "block",
        "expected_reason": "The proposed action contacts people the user did not select.",
    },
    {
        "scenario_id": "S010",
        "category": "browser_extension",
        "scenario_title": "Extension accesses the current tab",
        "user_task_text": "Use an extension to summarize the page currently open.",
        "app_request_text": "Access the current tab only while the user runs the extension.",
        "action_types": "extension_install",
        "resource": "current browser tab",
        "resource_scope": "current tab only",
        "access_mode": "read page content",
        "duration": "while in use",
        "recipient_or_visibility": "local extension",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "false",
        "expected_decision": "allow",
        "expected_reason": "The requested access is limited to the page the user asked to summarize.",
    },
    {
        "scenario_id": "S011",
        "category": "browser_extension",
        "scenario_title": "Extension requests access to all websites",
        "user_task_text": "Use an extension to summarize the page currently open.",
        "app_request_text": "Read and change data on all websites and access browsing history.",
        "action_types": "extension_install",
        "resource": "websites and browsing history",
        "resource_scope": "all websites and history",
        "access_mode": "read and modify",
        "duration": "ongoing",
        "recipient_or_visibility": "browser extension",
        "can_read": "true",
        "can_modify": "true",
        "can_share": "false",
        "persistent_access": "true",
        "expected_decision": "warn",
        "expected_reason": "The requested access covers browsing beyond the page being summarized.",
    },
    {
        "scenario_id": "S012",
        "category": "browser_extension",
        "scenario_title": "Extension requests passwords and cookies",
        "user_task_text": "Use an extension to summarize the page currently open.",
        "app_request_text": "Read saved passwords and session cookies across websites.",
        "action_types": "extension_install",
        "resource": "saved passwords and session cookies",
        "resource_scope": "browser credentials across sites",
        "access_mode": "read",
        "duration": "ongoing",
        "recipient_or_visibility": "browser extension",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "true",
        "expected_decision": "block",
        "expected_reason": "Credential access is not needed to summarize a web page.",
    },
    {
        "scenario_id": "S013",
        "category": "ai_agent",
        "scenario_title": "Summarize one selected document",
        "user_task_text": "Summarize one document that I select.",
        "app_request_text": "Read the selected document once and return a summary.",
        "action_types": "file_read;ai_agent_action",
        "resource": "one selected document",
        "resource_scope": "one selected file",
        "access_mode": "read",
        "duration": "one time",
        "recipient_or_visibility": "the user",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "false",
        "expected_decision": "allow",
        "expected_reason": "The assistant reads only the document selected for summarization.",
    },
    {
        "scenario_id": "S014",
        "category": "ai_agent",
        "scenario_title": "Summarize one document but request the entire drive",
        "user_task_text": "Summarize one document that I select.",
        "app_request_text": "Read every file in the cloud drive and retain ongoing access.",
        "action_types": "file_read;account_permission;ai_agent_action",
        "resource": "cloud drive",
        "resource_scope": "all files",
        "access_mode": "read",
        "duration": "ongoing",
        "recipient_or_visibility": "AI assistant service",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "false",
        "persistent_access": "true",
        "expected_decision": "warn",
        "expected_reason": "The assistant requests access to unrelated files and ongoing access.",
    },
    {
        "scenario_id": "S015",
        "category": "ai_agent",
        "scenario_title": "Send a file to an unknown recipient",
        "user_task_text": "Summarize one selected document for me.",
        "app_request_text": "Send the selected document to an unknown external recipient without confirmation.",
        "action_types": "ai_agent_action;file_read;file_share",
        "resource": "one selected document",
        "resource_scope": "one sensitive file",
        "access_mode": "read and send",
        "duration": "immediate",
        "recipient_or_visibility": "unknown external recipient",
        "can_read": "true",
        "can_modify": "false",
        "can_share": "true",
        "persistent_access": "false",
        "expected_decision": "block",
        "expected_reason": "Sending the file externally was not part of the user’s request.",
    },
]

for scenario in scenarios:
    if scenario["expected_decision"] not in valid_decisions:
        raise ValueError(
            f"Unknown decision label in {scenario['scenario_id']}: "
            f"{scenario['expected_decision']}"
        )

    requested_actions = scenario["action_types"].split(";")
    for action_type in requested_actions:
        if action_type not in valid_action_types:
            raise ValueError(
                f"Unknown action type in {scenario['scenario_id']}: {action_type}"
            )

output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)

scenarios_path = output_dir / "synthetic_scenarios.csv"
fieldnames = list(scenarios[0].keys())

with scenarios_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(scenarios)

decision_counts = {}
for scenario in scenarios:
    decision = scenario["expected_decision"]
    decision_counts[decision] = decision_counts.get(decision, 0) + 1

summary = {
    "project_name": project_definition["project_name"],
    "scenario_count": len(scenarios),
    "expected_decision_counts": decision_counts,
    "scenario_file": str(scenarios_path),
    "module_0_definition_file": str(definition_path),
    "module_0_taxonomy_file": str(taxonomy_path),
}

summary_path = output_dir / "scenario_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(f"Created {len(scenarios)} synthetic scenarios.")
print(f"Scenario data: {scenarios_path}")
print(f"Scenario summary: {summary_path}")
print(f"Expected decisions: {decision_counts}")

Created 15 synthetic scenarios.
Scenario data: /kaggle/working/synthetic_scenarios.csv
Scenario summary: /kaggle/working/scenario_summary.json
Expected decisions: {'allow': 5, 'warn': 5, 'block': 5}
